In [ ]:
import json, time
import numpy as np
from scipy import stats
from pathlib import Path
import anthropic

client = anthropic.Anthropic()   
CLAUDE_MODEL = 'claude-sonnet-4-6'
GRADES_DIR = Path('../models/grades')
GRADES_DIR.mkdir(parents=True, exist_ok=True)

SPECS = [
    ('../models/explanations/gemma_xgb_explanations.jsonl', 'tabular', 'gemma'),
    ('../models/explanations/gemma_gnn_explanations.jsonl', 'network',  'gemma'),
    ('../models/explanations/gemma_hybrid_explanations.jsonl', 'bimodal',  'gemma'),
    ('../models/explanations_test/deepseek_xgb_explanations.jsonl', 'tabular',  'deepseek'),
    ('../models/explanations/deepseek_gnn_explanations.jsonl', 'network',  'deepseek'),
    ('../models/explanations_test/deepseek_hybrid_explanations.jsonl', 'bimodal',  'deepseek'),
    ('../models/explanations_test/gemini_xgb_explanations.jsonl', 'tabular',  'gemini'),
    ('../models/explanations/gemini_gnn_explanations.jsonl', 'network',  'gemini'),
    ('../models/explanations_test/gemini_hybrid_explanations.jsonl', 'bimodal',  'gemini'),
]

METRIC_KEYS = ['Individual Feature Coverage', 'Individual Feature Consistency', 'Network Coverage', 'Network Consistency',]

def compute_confidence_interval(scores, confidence=0.95):
    if len(scores) == 0:
        return None, '± [NaN, NaN]'
    if len(scores) == 1:
        m = scores[0]
        return m, f'{m:.2f} (single sample - no CI possible)'
    m = np.mean(scores)
    if np.std(scores) == 0:
        return m, f'{m:.2f} ± 0.00 (95% CI = [{m:.2f}, {m:.2f}]) - identical values'
    h = stats.sem(scores) * stats.t.ppf((1 + confidence) / 2., len(scores) - 1)
    return m, f'{m:.2f} ± {h:.2f} (95% CI = [{m-h:.2f}, {m+h:.2f}])'

def print_ci(name, data):
    mean_score, ci_str = compute_confidence_interval(data)
    if mean_score is None:
        print(f'  {name}: No data available'); return
    if len(data) > 1 and np.std(data) > 0:
        h = stats.sem(data) * stats.t.ppf(0.975, len(data) - 1)
        lo = max(1.0, mean_score - h)
        hi = min(5.0, mean_score + h)
        bounded = (mean_score - h < 1.0) or (mean_score + h > 5.0)
        if bounded:
            print(f'{name}: {mean_score:.2f} ± {(hi-lo)/2:.2f} (95% CI = [{lo:.2f}, {hi:.2f}]) *bounded')
        else:
            print(f'{name}: {ci_str}')
    else:
        print(f'{name}: {ci_str}')

def build_prompt(explanation, insight, model_type):
    top_features = list(insight['top_features_impact'].keys())
    impacts = [
        f"{f}: {'increases' if v > 0 else 'decreases'} default risk (importance: {v:.3f})"
        for f, v in insight['top_features_impact'].items()
    ]
    approval_prob = 1 - insight.get('predicted_proba', 0.5)
    if model_type == 'hybrid':
        ctx = ('For HYBRID models: The explanation should demonstrate synthesis of individual borrower '
               'characteristics AND network/relational context. Credit for showing how personal factors '
               'interact with area/peer patterns. Hybrid explanations are expected to be more '
               'comprehensive than single-modality explanations.')
    else:
        ctx = f'For {model_type.upper()} models: Evaluate according to the model intended purpose and capabilities.'
    return (
        f'You are evaluating an AI explanation for loan default prediction. '
        f'The explanation comes from a {model_type.upper()} model.\n\n'
        f'EXPLANATION TO EVALUATE:\n"{explanation}"\n\n'
        f'GROUND TRUTH MODEL DATA:\n'
        f'- Model type: {model_type.upper()}\n'
        f'- Predicted approval probability: {approval_prob:.3f} ({approval_prob:.0%})\n'
        f'- Top individual features: {", ".join(top_features)}\n'
        f'- Feature impacts: {"; ".join(impacts)}\n\n'
        f'{ctx}\n\n'
        'EVALUATION CRITERIA:\n\n'
        '**Individual Feature Coverage (1-5):**\n'
        '- 5: Mentions all top 3 most important model features meaningfully\n'
        '- 4: Covers 2 of top 3 important features with good detail, minor gaps\n'
        '- 3: Covers some important features but misses key ones or includes less important ones\n'
        '- 2: Limited coverage of actually important model factors\n'
        '- 1: Focuses on irrelevant factors, ignores key model learnings\n\n'
        '**Individual Feature Consistency (1-5):**\n'
        '- 5: All mentioned individual factors have correct directional impact on approval probability\n'
        '- 4: Most individual factors directionally correct, minor inconsistencies\n'
        '- 3: Some correct directions but notable contradictions or unclear statements\n'
        '- 2: Frequent contradictions with actual model feature impacts\n'
        '- 1: Systematic misalignment with model learnings\n\n'
        '**Network Coverage (1-5):**\n'
        '- 5: Substantial network discussion with specific details (borrower counts, connection types, area patterns)\n'
        '- 4: Good network focus with adequate detail about connections or area factors\n'
        '- 3: Moderate network content - mentions connections but lacks depth\n'
        '- 2: Limited network discussion - brief mentions without meaningful detail\n'
        '- 1: No meaningful network or relational content\n\n'
        '**Network Consistency (1-5):**\n'
        '- 5: Accurate and logical network relationship descriptions with clear risk implications\n'
        '- 4: Generally correct network interpretations with sound reasoning\n'
        '- 3: Basic network understanding with mostly logical explanations\n'
        '- 2: Some network logic but with unclear or inconsistent elements\n'
        '- 1: Incorrect or contradictory network relationship descriptions\n\n'
        '**IMPORTANT NOTES:**\n'
        '- Customer-friendly language (e.g., "credit score" instead of "fico") is acceptable and preferred\n'
        '- Network/relationship factors may be described as "area patterns" or "similar borrowers"\n'
        '- For hybrid models, both individual and network factors should be well-represented\n\n'
        'Respond ONLY with a valid JSON object in this exact format:\n'
        '{\n'
        '"Individual Feature Coverage": <integer 1-5>,\n'
        '"Individual Feature Consistency": <integer 1-5>,\n'
        '"Network Coverage": <integer 1-5>,\n'
        '"Network Consistency": <integer 1-5>\n'
        '}'
    )

def grade_explanation(explanation, insight, model_type, max_retries=5):
    if not explanation or not explanation.strip():
        print('  [skip] empty explanation'); return None
    if not insight or 'top_features_impact' not in insight:
        print('  [skip] invalid insight'); return None
    prompt = build_prompt(explanation, insight, model_type)
    for attempt in range(1, max_retries + 1):
        try:
            response = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=256, temperature=0.1,
                system='You are an expert evaluator trained to assess ML explanations. Respond ONLY with a valid JSON object.',
                messages=[{'role': 'user', 'content': prompt}],
            )
            raw = response.content[0].text.strip()
            if raw.startswith('```'):
                raw = raw.split('```')[1]
                if raw.startswith('json'): raw = raw[4:]
            grades = json.loads(raw)
            if not all(k in grades for k in METRIC_KEYS):
                raise ValueError(f'Missing keys: {grades}')
            for k in METRIC_KEYS:
                if not isinstance(grades[k], (int, float)) or not (1 <= grades[k] <= 5):
                    raise ValueError(f'Invalid score for {k}: {grades[k]}')
            return {k: int(grades[k]) for k in METRIC_KEYS}
        except (json.JSONDecodeError, ValueError, anthropic.APIStatusError) as e:
            wait = min(4 * 2 ** (attempt - 1), 60)
            time.sleep(wait)
    print('Max retries exceeded'); return None

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def load_existing_grades(grades_path):
    if not grades_path.exists():
        return {}, []
    records = [json.loads(l) for l in open(grades_path) if l.strip()]
    return {r['record_index']: r for r in records}, records

def run_grading(path, model_type, llm_name):
    grades_path = GRADES_DIR / f'{llm_name}_{model_type}_grades.jsonl'

    print(f"\n{'='*60}")
    print(f'LLM: {llm_name.upper()} | Model type: {model_type.upper()}')
    print(f'Input : {path}')
    print(f'Output: {grades_path}')

    records = load_jsonl(path)
    already_graded, existing_records = load_existing_grades(grades_path)

    if already_graded:
        print(f'Resuming -- {len(already_graded)}/{len(records)} already graded, skipping those.')
    else:
        print(f'Loaded {len(records)} records -- grading from scratch.')

    scores = {k: [] for k in METRIC_KEYS}
    for r in existing_records:         
        for k in METRIC_KEYS:
            if k in r: scores[k].append(r[k])

    with open(grades_path, 'a') as out_f:
        for i, rec in enumerate(records):
            if i in already_graded:
                print(f'  [{i+1}/{len(records)}] already graded -- skipping')
                continue
            explanation = rec.get('explanation') or rec.get('generated_explanation', '')
            insight = rec.get('insight') or rec.get('shap_insight') or {}
            print(f'Grading {i+1}/{len(records)}...', end=' ', flush=True)
            result = grade_explanation(explanation, insight, model_type)
            if result:
                for k in METRIC_KEYS: scores[k].append(result[k])
                print(' | '.join(f'{k[:4]}: {result[k]}' for k in METRIC_KEYS))
                out_record = {'record_index': i, 'llm': llm_name, 'model_type': model_type, **result}
                out_f.write(json.dumps(out_record) + '\n')
                out_f.flush()       
            else:
                print('skipped')

    print(f'\n### 95% Confidence Intervals -- {llm_name.upper()} / {model_type.upper()} ###')
    for name in METRIC_KEYS:
        print_ci(name, scores[name])
    return scores

all_results = {}
for path, model_type, llm_name in SPECS:
    if not Path(path).exists():
        print(f'\nFile not found, skipping: {path}')
        continue
    all_results[f'{llm_name}_{model_type}'] = run_grading(path, model_type, llm_name)

print(f"\n{'='*60}")
print('Aggregate 95% CIs')
print(f"{'='*60}")
agg = {k: [] for k in METRIC_KEYS}
for scores in all_results.values():
    for k in METRIC_KEYS: agg[k].extend(scores[k])
for name in METRIC_KEYS:
    print_ci(name, agg[name])

print(f'\nAll grades saved to: {GRADES_DIR.resolve()}')
